# Previsão do tempo de ciclo — nova camada ouro

Fonte única: `s3://delivery-ouro-p/ciclo_entregas/` (tabela criada como
`delivery_prata_db.ouro_ciclo_entregas`). Execute em ordem no kernel do `venv`.
Alvo: `order_metric_cycle_time`, interpretado em minutos. Seleção: entrega
`DELIVERED` e ciclo entre **20 e 120 minutos**, incluindo os limites.

Este é um recorte experimental, não uma regra que prova que todas as outras
durações sejam incorretas. A avaliação representa somente pedidos dessa faixa.
Os resultados reais serão produzidos ao executar; saídas antigas foram limpas.


## 1. Configuração

Leitura direta de Parquet no S3: sem consulta ao Athena, Glue ou à prata.
O perfil local `default` deve autenticar como `docas32`; não inserir chaves aqui.


In [ ]:
"""Previsao do tempo de ciclo de pedidos entregues entre 20 e 120 minutos."""

from pathlib import Path

import awswrangler as wr
import boto3
import joblib
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

REGIAO = "sa-east-1"
PERFIL_AWS = "default"
USUARIO_AWS = "docas32"
CAMADA_OURO = "s3://delivery-ouro-p/ciclo_entregas/"
LIMITE_MINUTOS_MIN = 20
LIMITE_MINUTOS_MAX = 120
RESULTADOS = Path(r'C:\Users\libai\Documents\aws-delivery-eta-prediction') / "resultados"
ALVO = "order_metric_cycle_time"
NUMERICAS = ["store_latitude", "store_longitude", "order_amount", "hora", "dia_semana", "mes"]
ATRIBUTOS = ["store_id", *NUMERICAS]


from IPython.display import display


## 2. Verificar identidade AWS

O nome do perfil não é o nome do usuário IAM. A verificação STS evita executar
com outro usuário. São necessárias permissões S3 de listar e ler `ciclo_entregas/`.


In [ ]:
# Seleciona explicitamente o perfil padrao do arquivo de credenciais AWS.
# O nome do perfil (default) e diferente do nome do usuario IAM (docas32).
sessao = boto3.Session(profile_name=PERFIL_AWS, region_name=REGIAO)
# STS confirma a identidade real; mudar somente o nome do usuario no codigo
# nao troca as credenciais. Interrompemos antes de ler S3 se o usuario divergir.
identidade = sessao.client("sts").get_caller_identity()["Arn"]
if not identidade.endswith(f":user/{USUARIO_AWS}") and not (
    ":user/" in identidade and identidade.rsplit("/", 1)[-1] == USUARIO_AWS
):
    raise RuntimeError(
        f"O perfil {PERFIL_AWS!r} autentica como {identidade}, mas o usuario "
        f"esperado e {USUARIO_AWS}. Configure localmente as credenciais de "
        f"{USUARIO_AWS} no perfil default (aws configure --profile default) "
        "e execute novamente. Nao coloque chaves neste codigo ou no chat."
    )
print(f"Usuario AWS confirmado: {USUARIO_AWS}", flush=True)


## 3. Ler a ouro consolidada

A construção da ouro selecionou pedidos com pelo menos uma entrega `DELIVERED`,
consolidando a chave de entregas antes da junção. Aqui verificamos que `order_id`
é preenchido e único. `order_status` é mantido para auditoria, mas não exigimos
`FINISHED`: o filtro aprovado é o status de entrega.


In [ ]:
def carregar_dados(sessao):
    # A nova ouro ja consolida pedidos, lojas e a existencia de entrega DELIVERED.
    # Lemos somente esta fonte; nao precisamos de acesso a prata ou ao Glue.
    # Parquet e lido pelo conteudo, inclusive arquivos CTAS sem extensao .parquet.
    print("Lendo a camada ouro...", flush=True)
    dados = wr.s3.read_parquet(
        path=CAMADA_OURO, boto3_session=sessao,
        columns=["order_id", "delivery_order_id", "store_id", "store_latitude",
                 "store_longitude", "order_amount", "order_status", "delivery_status",
                 "order_moment_created", "order_moment_finished", ALVO],
    )
    if dados["order_id"].isna().any() or dados["order_id"].duplicated().any():
        raise ValueError("A ouro deve ter order_id preenchido e uma unica linha por pedido.")
    print(f"Pedidos na ouro: {len(dados):,}")
    return dados

dados_brutos = carregar_dados(sessao)
display(dados_brutos.head())


## 4. Diagnóstico antes do filtro

Na consulta compartilhada, a ouro tinha 343.936 pedidos: 688 alvos ausentes,
8.593 abaixo de 20, 314.214 entre 20 e 120 e 20.441 acima de 120 minutos.
Esses números são uma referência; a célula abaixo recalcula a distribuição.

`order_moment_delivered` não é usado: estava ausente na maioria da prata.
`NaT` significa data ausente e não se resolve apenas convertendo o tipo.
A métrica de ciclo foi adotada como alvo; finalização administrativa não é
tratada como horário de entrega.


In [ ]:
display(dados_brutos.dtypes.to_frame("tipo"))
display(dados_brutos.groupby(["delivery_status", "order_status"], dropna=False).size().to_frame("pedidos"))
ciclo_bruto = pd.to_numeric(dados_brutos[ALVO], errors="coerce")
faixas = pd.Series("alvo ausente ou não finito", index=dados_brutos.index)
finito = ciclo_bruto.notna() & np.isfinite(ciclo_bruto)
faixas.loc[finito & ciclo_bruto.lt(20)] = "abaixo de 20"
faixas.loc[finito & ciclo_bruto.between(20, 120)] = "entre 20 e 120"
faixas.loc[finito & ciclo_bruto.gt(120)] = "acima de 120"
resumo_faixas = faixas.value_counts().to_frame("pedidos")
resumo_faixas["percentual"] = resumo_faixas["pedidos"] / len(dados_brutos) * 100
display(resumo_faixas)


## 5. Preparar a amostra

Decisão: filtrar `DELIVERED`, alvo finito entre 20 e 120 e criação válida.
Não imputamos o alvo com moda e não alteramos os arquivos da ouro.
Valores ausentes nos atributos numéricos serão imputados pela mediana do treino.
Os horários devem seguir a mesma convenção de fuso da fonte.

Para aproximar quando o alvo estaria disponível, usamos o instante mais tardio
entre criação + ciclo e finalização observada, quando presente. Criação + ciclo
é uma estimativa, não uma recuperação do horário real da entrega. Não temos
o horário de ingestão para provar disponibilidade operacional exata.


In [ ]:
def preparar_dados(dados):
    dados = dados.copy()
    for coluna in ["order_moment_created", "order_moment_finished"]:
        dados[coluna] = pd.to_datetime(dados[coluna], errors="coerce")
    # Alvo vem pronto da ouro: nao depende de order_moment_delivered, muito ausente.
    dados[ALVO] = pd.to_numeric(dados[ALVO], errors="coerce")
    entregues = dados["delivery_status"].eq("DELIVERED").fillna(False)
    finitos = dados[ALVO].notna() & np.isfinite(dados[ALVO])
    faixa = dados[ALVO].between(LIMITE_MINUTOS_MIN, LIMITE_MINUTOS_MAX)
    # Recorte aprovado do experimento, com limites inclusivos. Nao imputamos
    # duracoes pela moda nem alteramos os arquivos completos da ouro.
    print(f"Alvos ausentes ou nao finitos: {(~finitos).sum():,}")
    print(f"Abaixo de 20 minutos: {(finitos & dados[ALVO].lt(LIMITE_MINUTOS_MIN)).sum():,}")
    print(f"Acima de 120 minutos: {(finitos & dados[ALVO].gt(LIMITE_MINUTOS_MAX)).sum():,}")
    print(f"Status diferente de DELIVERED: {(~entregues).sum():,}")
    print(f"Instante de criacao ausente: {dados['order_moment_created'].isna().sum():,}")
    validos = entregues & finitos & faixa & dados["order_moment_created"].notna()
    print(f"Pedidos para o experimento: {validos.sum():,} de {len(dados):,}")
    dados = dados.loc[validos].sort_values("order_moment_created").copy()
    # Criacao + ciclo e uma estimativa de fim do ciclo, nao uma data observada.
    # Para o corte temporal usamos o mais tardio entre essa estimativa e a
    # finalizacao observada, quando existe, como aproximacao conservadora da
    # disponibilidade do alvo. O instante real de ingestao nao esta na ouro.
    fim_estimado = dados["order_moment_created"] + pd.to_timedelta(dados[ALVO], unit="m")
    dados["disponibilidade_alvo"] = pd.concat(
        [fim_estimado, dados["order_moment_finished"]], axis=1
    ).max(axis=1)
    dados["store_id"] = dados["store_id"].fillna("desconhecida").astype(str)
    dados["hora"] = dados["order_moment_created"].dt.hour
    dados["dia_semana"] = dados["order_moment_created"].dt.dayofweek
    dados["mes"] = dados["order_moment_created"].dt.month
    for coluna in NUMERICAS:
        dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce").replace([np.inf, -np.inf], np.nan)
    return dados

dados = preparar_dados(dados_brutos)
display(dados.head())


## 6. Análise do recorte

Inspecione distribuição, ausências e volumes por loja e horário.
Associações são descritivas e não demonstram causalidade.


In [ ]:
display(dados[ALVO].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame("minutos"))
display(dados[ATRIBUTOS].isna().mean().mul(100).to_frame("ausentes_pct"))
display(dados.groupby("store_id")[ALVO].agg(pedidos="size", mediana="median", media="mean").sort_values("pedidos", ascending=False).head(20))
display(dados.groupby("hora")[ALVO].agg(pedidos="size", mediana="median"))


## 7. Separação temporal

Aproximadamente 20% dos pedidos mais recentes são reservados para teste.
Empates no instante de criação ficam do mesmo lado. No treino, também exigimos
que a aproximação de disponibilidade do alvo anteceda o corte. Os mínimos de
volume são verificações operacionais, não garantias estatísticas.


In [ ]:
if len(dados) < 50:
    raise ValueError("Sao necessarios pelo menos 50 pedidos validos para esta avaliacao.")
# Os 20% mais recentes ficam para teste, simulando previsoes futuras.
corte = dados.iloc[int(len(dados) * 0.8)]["order_moment_created"]
# Evita usar alvos ainda indisponiveis no corte segundo a aproximacao acima.
treino = dados.loc[(dados["order_moment_created"] < corte) & (dados["disponibilidade_alvo"] < corte)]
teste = dados.loc[dados["order_moment_created"] >= corte]
if len(treino) < 30 or len(teste) < 10:
    raise ValueError("O corte temporal nao produziu treino e teste suficientes.")


print(f"Treino: {len(treino):,}; teste: {len(teste):,}; corte: {corte}")


## 8. Modelo e prevenção de vazamento

Atributos: loja, coordenadas, valor do pedido, hora, dia da semana e mês.
Status, alvo e horários posteriores à criação não entram no modelo.
Pipeline aprende imputação e categorias apenas no treino. One-hot agrupa lojas
raras e limita categorias para controlar memória. Histogram Gradient Boosting
é o ponto de partida para relações não lineares; não ajustamos parâmetros no teste.


In [ ]:
# Apenas atributos disponiveis na criacao entram no modelo.
# Tempo de cozinha e horarios posteriores causariam vazamento de informacao.
tratamento = ColumnTransformer([
    ("numeros", SimpleImputer(strategy="median", keep_empty_features=True), NUMERICAS),
    ("loja", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                           max_categories=100, sparse_output=False), ["store_id"]),
])
modelo = Pipeline([
    ("tratamento", tratamento),
    ("regressor", HistGradientBoostingRegressor(max_iter=150, random_state=42)),
])
modelo.fit(treino[ATRIBUTOS], treino[ALVO])
previsoes = np.maximum(modelo.predict(teste[ATRIBUTOS]), 0)


## 9. Avaliação

MAE mede o erro absoluto médio em minutos; RMSE penaliza mais os erros grandes.
Compare com a referência que prevê a mediana do treino. As métricas valem apenas
para ciclos entre 20 e 120; não representam cancelamentos nem a operação completa.
Não limitamos previsões artificialmente à faixa para melhorar as métricas.


In [ ]:
# A mediana e uma referencia simples que o modelo deve superar.
baseline = DummyRegressor(strategy="median").fit(treino[ATRIBUTOS], treino[ALVO])
mae_base = mean_absolute_error(teste[ALVO], baseline.predict(teste[ATRIBUTOS]))
mae = mean_absolute_error(teste[ALVO], previsoes)
rmse = np.sqrt(mean_squared_error(teste[ALVO], previsoes))
print(f"Treino: {len(treino):,}; teste: {len(teste):,}; corte: {corte}")
print(f"MAE: {mae:.2f} min | RMSE: {rmse:.2f} min | MAE da mediana: {mae_base:.2f} min")
if mae >= mae_base:
    print("O modelo ainda nao supera a mediana; revise atributos e qualidade dos dados.")


## 10. Exportação local

Arquivos: `previsoes_ciclo_teste.csv` e `modelo_tempo_ciclo.joblib` em `resultados/`.
Salvamos o modelo avaliado, sem retreinar com o teste. O artefato registra fonte,
status, alvo, faixa e métricas. Não há escrita nos buckets de origem.


In [ ]:
RESULTADOS.mkdir(parents=True, exist_ok=True)
saida = teste[["order_id", "order_moment_created", ALVO]].copy()
saida["previsao_minutos"] = previsoes
saida["erro_absoluto_minutos"] = abs(saida[ALVO] - previsoes)
saida.to_csv(RESULTADOS / "previsoes_ciclo_teste.csv", index=False)
# Salva o modelo avaliado, sem retreinar com os dados reservados para teste.
joblib.dump({"modelo": modelo, "atributos": ATRIBUTOS, "alvo": ALVO,
             "corte_temporal": corte, "mae_minutos": mae,
             "rmse_minutos": rmse, "mae_baseline_minutos": mae_base,
             "fonte": CAMADA_OURO, "status": "DELIVERED",
             "faixa_minutos": [LIMITE_MINUTOS_MIN, LIMITE_MINUTOS_MAX]},
            RESULTADOS / "modelo_tempo_ciclo.joblib")
print(f"Modelo e previsoes salvos em: {RESULTADOS}")

display(saida.head())
display(saida.nlargest(10,"erro_absoluto_minutos"))


## 11. Exemplo de previsão

A célula reutiliza atributos de um pedido do teste para demonstrar o formato.
Na operação, substitua pelos atributos de um novo pedido na criação. O modelo
é condicionado ao recorte escolhido, sem garantir que o pedido pertencerá à faixa.


In [ ]:
novo_pedido = teste[ATRIBUTOS].iloc[[0]].copy()
print(f"Ciclo previsto: {max(float(modelo.predict(novo_pedido)[0]), 0):.1f} minutos")
